In [2]:
# Import our newly created DataLoader module
from src.data_loader import DataLoader

# Initialize the loader
loader = DataLoader(data_dir='data', p_threshold=5)

# Load the combined data from all files
df = loader.load_all_data()

print(f"Success! Loaded {len(df)} rows from {df['source_file'].nunique()} files.")

# Display the first 5 rows
display(df.head())

Success! Loaded 30 rows from 8 files.


,m,p,n_LU,N_C,R,source_file
0,10,50,66.0,32,1.836697,Ex_1.csv
1,10,105,140.0,32,1.727009,Ex_1.csv
2,10,196,260.0,32,1.671914,Ex_1.csv
3,10,1210,1530.0,32,1.321586,Ex_1.csv
4,30,88,192.0,32,0.744993,Ex_3.csv


In [3]:
# Import all our custom built modules
from src.data_loader import DataLoader
from src.preprocessing import StandardScalerStrategy
from src.models import RegressionModelFactory
from src.evaluator import ModelEvaluator
from src.plotter import ResultPlotter
import pandas as pd

# 1. Initialize Components
print("Initializing pipeline components...")
loader = DataLoader(data_dir='data', p_threshold=5)
scaler_X = StandardScalerStrategy()
scaler_y = StandardScalerStrategy()

# Factory: 3 hidden layers (128, 64, 32 neurons), Adam optimizer
model_factory = RegressionModelFactory(input_dim=4, hidden_layers=[128, 64, 32], learning_rate=0.001)
evaluator = ModelEvaluator(model_factory, scaler_X, scaler_y)
plotter = ResultPlotter(save_dir='results')

# 2. Load Data
print("\nLoading and preparing data...")
X, y, sources = loader.get_X_y()

# 3. Evaluate Model (K-Fold Validation)
# Note: epochs=50 to make it run reasonably fast for testing. You can increase it to 100 or 200 later.
results = evaluator.evaluate(X, y, sources, n_splits=5, epochs=50, batch_size=32)

# 4. Generate Visualizations from the captured Fold 1 data
print("\nGenerating charts...")
fold_data = results['fold_1_data']

plotter.plot_learning_curve(fold_data['history'])
plotter.plot_predictions_per_file(fold_data['y_true'], fold_data['y_pred'], fold_data['sources'])

# 5. Final Report (Displaying global metrics as a DataFrame)
print("\n--- FINAL REPORT ---")
report_df = pd.DataFrame([{
    'Metric': 'Mean Squared Error (MSE)',
    'Average': round(results['mean_mse'], 4),
    'Std Dev (+/-)': round(results['std_mse'], 4)
}, {
    'Metric': 'Mean Absolute Error (MAE)',
    'Average': round(results['mean_mae'], 4),
    'Std Dev (+/-)': round(results['std_mae'], 4)
}])

display(report_df)
print("\nPipeline execution finished successfully! Check the 'results/' folder for your charts.")

Initializing pipeline components...

Loading and preparing data...
Starting 5-Fold Cross-Validation...
--- Training Fold 1/5 ---
Fold 1 - Real MSE: 0.8963, Real MAE: 0.6094
--- Training Fold 2/5 ---
Fold 2 - Real MSE: 0.2298, Real MAE: 0.3153
--- Training Fold 3/5 ---
Fold 3 - Real MSE: 0.7442, Real MAE: 0.4749
--- Training Fold 4/5 ---
Fold 4 - Real MSE: 2.1044, Real MAE: 1.0068
--- Training Fold 5/5 ---
Fold 5 - Real MSE: 0.2987, Real MAE: 0.4813

Cross-Validation Complete!
Average Real MSE: 0.8547 (+/- 0.6746)
Average Real MAE: 0.5775 (+/- 0.2340)

Generating charts...
Saved learning curve to: results/global_learning_curve.png
Saved 4 scatter plots to the 'results' directory.

--- FINAL REPORT ---


,Metric,Average,Std Dev (+/-)
0,Mean Squared Error (MSE),0.8547,0.6746
1,Mean Absolute Error (MAE),0.5775,0.2340



Pipeline execution finished successfully! Check the 'results/' folder for your charts.


In [ ]:
import itertools

print("--- PHASE 7: Supercomputer Extrapolation ---")

# 1. Train the final "Best Model" on ALL available data for maximum accuracy
print("Training final global model on all data...")
final_model = model_factory.create_model()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1))

final_model.fit(X_scaled, y_scaled, epochs=100, batch_size=32, verbose=0)
print("Final model trained successfully.")

# 2. Extract unique combinations of (m, p, n_LU) from the original dataset
df_original = pd.DataFrame(X, columns=['m', 'p', 'n_LU', 'N_C'])
unique_combinations = df_original[['m', 'p', 'n_LU']].drop_duplicates()

# 3. Define the new hardware environments (N_C) requested by the professor
future_nc_values = [64, 128, 256, 512, 1024]

# 4. Generate all combinations of existing (m, p, n_LU) with the new N_C values
future_data = []
for _, row in unique_combinations.iterrows():
    for nc in future_nc_values:
        future_data.append([row['m'], row['p'], row['n_LU'], nc])
        
df_future = pd.DataFrame(future_data, columns=['m', 'p', 'n_LU', 'N_C'])
print(f"Generated {len(df_future)} hypothetical scenarios for prediction.")

# 5. Predict using the trained model
X_future_scaled = scaler_X.transform(df_future.values)
predictions_scaled = final_model.predict(X_future_scaled, verbose=0)

# Inverse transform to get real 'R' values
predictions_real = scaler_y.inverse_transform(predictions_scaled)
df_future['Predicted_R'] = predictions_real

# 6. Save the final predictions to a CSV file
output_path = 'results/supercomputer_predictions.csv'
df_future.to_csv(output_path, index=False)

print(f"Predictions successfully saved to: {output_path}")
display(df_future.head(10))